# Notebook 01 — Ingest Raw Data (Bronze Layer)

Pull live data from the Fantasy Premier League API and land it as raw Delta tables
in the `fpl_project.bronze` schema. Each table gets two metadata columns
(`_ingestion_ts`, `_source`) so we can trace every row back to a specific API call.

**Data flow**: FPL REST API → NDJSON files on Volumes → Delta managed tables

In [ ]:
import requests
import json
from pyspark.sql import functions as F
from datetime import datetime, timezone
import time

# Base URL for the free, unauthenticated FPL API
BASE_URL = "https://fantasy.premierleague.com/api"

# Capture a single timestamp so every table written in this run shares the same value
INGESTION_TS = datetime.now(timezone.utc).isoformat()


def fetch_api(endpoint):
    """Fetch JSON from the FPL API and return as a Python dict.

    Raises an HTTPError on 4xx/5xx responses so failures surface immediately.
    """
    response = requests.get(f"{BASE_URL}/{endpoint}")
    response.raise_for_status()
    return response.json()

## Ingest Bootstrap-Static

The `/bootstrap-static/` endpoint returns a single ~1.3 MB JSON payload containing
four nested arrays: `elements` (players), `teams`, `events` (gameweeks), and
`element_types` (positions). We split them into separate Delta tables.

In [ ]:
bootstrap = fetch_api("bootstrap-static/")

# Map each nested array to a meaningful table name
datasets = {
    "players": bootstrap["elements"],      # ~700 players with 50+ stat columns
    "teams": bootstrap["teams"],            # 20 Premier League clubs
    "events": bootstrap["events"],          # 38 gameweeks
    "positions": bootstrap["element_types"],# GKP, DEF, MID, FWD
}

for name, data in datasets.items():
    # Write NDJSON (one JSON object per line) to a Unity Catalog Volume,
    # then read it back as a Spark DataFrame so Spark infers the schema.
    path = f"/Volumes/fpl_project/bronze/raw_files/{name}.json"

    with open(path, "w") as f:
        for record in data:
            f.write(json.dumps(record) + "\n")

    df = spark.read.json(path) \
        .withColumn("_ingestion_ts", F.lit(INGESTION_TS)) \
        .withColumn("_source", F.lit("bootstrap-static"))

    # Append mode: each run adds a new snapshot so we keep history
    df.write.mode("append").format("delta") \
        .saveAsTable(f"fpl_project.bronze.{name}")

print("Bootstrap ingestion complete.")

## Ingest Fixtures

The `/fixtures/` endpoint returns every Premier League match for the season,
including scores, kickoff times, and FPL difficulty ratings (1-5).

In [ ]:
fixtures = fetch_api("fixtures/")

path = "/Volumes/fpl_project/bronze/raw_files/fixtures.json"
with open(path, "w") as f:
    for record in fixtures:
        f.write(json.dumps(record) + "\n")

fixtures_df = spark.read.json(path) \
    .withColumn("_ingestion_ts", F.lit(INGESTION_TS)) \
    .withColumn("_source", F.lit("fixtures"))

fixtures_df.write.mode("append").format("delta") \
    .saveAsTable("fpl_project.bronze.fixtures")

print("Fixtures ingestion complete.")

## Ingest Player Histories

The `/element-summary/{id}/` endpoint must be called once per player, so this
cell loops through every player ID with a 0.5 s delay to respect API rate limits.
Each player's gameweek-by-gameweek history is collected into a single Delta table.

In [ ]:
# Collect all distinct player IDs from the bronze players table we just wrote
player_ids = [
    row.id for row in
    spark.table("fpl_project.bronze.players")
    .select("id").distinct().collect()
]

# Fetch each player's gameweek history one at a time
all_histories = []
for pid in player_ids:
    try:
        summary = fetch_api(f"element-summary/{pid}/")
        for h in summary.get("history", []):
            h["player_id"] = pid          # tag each record with its player
            all_histories.append(h)
        time.sleep(0.5)                   # rate-limit: ~2 req/sec
    except Exception as e:
        print(f"Failed for player {pid}: {e}")

if all_histories:
    path = "/Volumes/fpl_project/bronze/raw_files/player_history.json"
    with open(path, "w") as f:
        for record in all_histories:
            f.write(json.dumps(record) + "\n")

    history_df = spark.read.json(path) \
        .withColumn("_ingestion_ts", F.lit(INGESTION_TS))
    history_df.write.mode("append").format("delta") \
        .saveAsTable("fpl_project.bronze.player_history")

print(f"Ingested history for {len(all_histories)} player-gameweek records.")